In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "aacPass2025"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso_Salvare_Logo.png' 
try:
    encoded_image = base64.b64encode(open(image_filename, 'rb').read())
    logo_element = html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), 
                            style={'height': '100px', 'width': 'auto'})
except:
    #Incase image doesnt load 
    logo_element = html.H2('Grazioso Salvare')

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Div(logo_element,style={'float': 'left', 'margin' : '10px'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Center(html.H2('Filip Arghir')),
    html.Hr(),
    html.Div(
        
#FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.

    ),
    html.Div([
        html.Label('Select Rescue Type filter:', style={'fontWeight': 'bold'}),
        dcc.RadioItems(
            id ='filter-type',
            options=[
                {'label': 'Adoption', 'value' : 'Adoption'},
                {'label': 'Transfer', 'value' : 'Transfer'},
                {'label': 'Euthanasia', 'value' : 'Euthanasia'},
                {'label': 'Return to Owner', 'value' : 'Return to Owner'},
                {'label': 'Reset(Show All)', 'value' : 'reset'}
            ],
            value='reset',
            style={'marginBottom': '20px'},
            inline=True,
            inputStyle={"margin-right": "5px", "margin-left": "20px"}
        ),
    ], style={'margin': '20px'}),
    
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here
        page_size=10,                   #pagination
        page_action='native', 
        sort_action='native',           #sorting
        filter_action='native',         #filtering
        row_selectable='single',        #map selection
        selected_rows=[0],               #default selected row

                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
## FIX ME Add code to filter interactive data table with MongoDB queries
#
#        
#        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
#        data=df.to_dict('records')
#       
#       
#        return (data,columns)
    if filter_type == 'Adoption':
        query = {'outcome_type': "Adoption"}
    elif filter_type == 'Transfer':
        query = {'outcome_type': "Transfer"}
    elif filter_type == 'Euthanasia':
        query = {'outcome_type': "Euthanasia"}
    elif filter_type == 'Return to Owner':
        query = {'outcome_type': "Return to Owner"}
    else:
        query = {}
      
    
    df_filtered = pd.DataFrame.from_records(db.read(query))
    if'_id' in df_filtered.columns:
        df_filtered.drop(columns=['_id'], inplace=True)
    return df_filtered.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),])
def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #
    if viewData is None or len(viewData) == 0:
        return[]
    
    dff = pd.DataFrame.from_dict(viewData)
    if 'breed' in dff.columns:
        breed_counts = dff['breed'].value_counts().head(10)
        fig = px.pie(values=breed_counts.values, 
                     names=breed_counts.index, 
                     title='Preferred Animals', 
                     height=400)
        return[dcc.Graph(figure=fig)]
    return[]

    #return [
    #    dcc.Graph(            
    #        figure = px.pie(df, names='breed', title='Preferred Animals')
    #    )    
    #]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return[]
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    # FIX 2: Check if data exists and wrap in try-except
    if viewData is None or len(viewData) == 0:
        return []
    
    try:
        dff = pd.DataFrame.from_dict(viewData)
        
        if index is None:
            row = 0
        else:
            row = index[0]
        
        return[
            dl.Map(style={'width': '1000px', 'height': '500px'},
                  center=[30.75, -97.48], zoom = 10, children=[
                  dl.TileLayer(id="base-layer-id"),
                  dl.Marker(position=[dff.iloc[row, 13], dff.iloc[row,14]],
                            children=[
                            dl.Tooltip(dff.iloc[row, 4]), 
                            dl.Popup([
                                html.H1("Animal Name"), 
                                html.P(dff.iloc[row, 9])
                            ])
                        ])                                                                             
                ])
        ]
    except:
        return []  # Return empty if any error
  


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(port=8070) 

Dash app running on https://eliteminimum-audiocanvas-3000.codio.io/proxy/8070/
